In [16]:
import os
import numpy as np
import pandas as pd
import scipy.signal as ss
import mne
import antropy as ant

from mne.decoding import CSP

In [5]:
df = pd.read_csv('dataset_BCI_completo.csv')

df.head()

,Sujeto,Condicion,Pot_Mu_C3,Pot_Mu_C4,Pot_Mu_Cz,Pot_Beta_C3,Pot_Beta_C4,Pot_Beta_Cz,Coh_Mu_C3_C4,Coh_Mu_C3_Cz,Coh_Mu_C4_Cz,Coh_Beta_C3_C4,Coh_Beta_C3_Cz,Coh_Beta_C4_Cz,Tipo
0,S001,Derecha,3.306863e-11,2.853744e-11,3.415199e-11,8.826270e-12,8.399120e-12,9.569256e-12,0.607774,0.842584,0.820079,0.550151,0.826825,0.683419,Imaginación Motora
1,S001,Reposo,3.105931e-11,2.168272e-11,3.053431e-11,1.279435e-11,7.969746e-12,1.130518e-11,0.702683,0.817302,0.850840,0.669249,0.849468,0.817461,Imaginación Motora
2,S001,Izquierda,3.244346e-11,2.711707e-11,3.483905e-11,8.441985e-12,6.591918e-12,7.897998e-12,0.736120,0.829747,0.922998,0.548569,0.725583,0.784959,Imaginación Motora
3,S001,Reposo,3.012843e-11,2.871449e-11,3.223302e-11,8.385304e-12,7.612996e-12,8.640117e-12,0.524534,0.806225,0.752072,0.567207,0.786638,0.717071,Imaginación Motora
4,S001,Izquierda,3.225722e-11,1.758506e-11,2.746542e-11,9.115796e-12,7.481623e-12,9.094855e-12,0.760350,0.895360,0.838313,0.546283,0.792682,0.728047,Imaginación Motora


In [7]:
ruta_base='Sujetos'

In [25]:
def obtener_metricas_eeg(
        raw,
        canales=['C3','Cz','C4']
    ):

    raw = raw.copy()

    raw.pick(canales)

    Fs = raw.info['sfreq']

    datos = raw.get_data()

    resultados={}

    # PSD + MU + BETA + ERD/ERS
    for i,canal in enumerate(canales):

        señal=datos[i]

        f,Pxx = ss.welch(

            señal,

            fs=Fs,

            window='hann',

            nperseg=int(2*Fs)
        )

        idx_mu=(f>=8)&(f<=13)

        idx_beta=(f>=14)&(f<=30)

        mu=np.mean(Pxx[idx_mu])

        beta=np.mean(Pxx[idx_beta])

        total=np.mean(Pxx)

        ERD_mu=((mu-total)/total)*100
        ERD_beta=((beta-total)/total)*100

        # ENTROPÍA ESPECTRAL (Proyecto 2)
        entropy = ant.spectral_entropy(

            señal,

            sf=Fs,

            method='welch',

            normalize=True
        )

        # HJORTH (Proyecto 2)
        actividad=np.var(señal)

        movilidad,complejidad=ant.hjorth_params(
            señal
        )

        resultados[canal]={

            'PSD':Pxx,
            'frecuencias':f,

            'mu':mu,
            'beta':beta,

            'ERD_mu':ERD_mu,
            'ERD_beta':ERD_beta,

            'spectral_entropy':entropy,

            'hjorth_activity':actividad,
            'hjorth_mobility':movilidad,
            'hjorth_complexity':complejidad
        }

    # COHERENCIA
    pares=[

        ('C3','C4'),
        ('C3','Cz'),
        ('C4','Cz')

    ]

    coherencias={}

    for c1,c2 in pares:

        idx1=canales.index(c1)
        idx2=canales.index(c2)

        f_coh,Cxy = ss.coherence(

            datos[idx1],

            datos[idx2],

            fs=Fs,

            nperseg=int(2*Fs)
        )

        coh_mu=np.mean(
            Cxy[(f_coh>=8)&(f_coh<=13)]
        )

        coh_beta=np.mean(
            Cxy[(f_coh>=14)&(f_coh<=30)]
        )

        coherencias[f'{c1}-{c2}']={

            'coh_mu':coh_mu,
            'coh_beta':coh_beta
        }

    resultados['coherencia']=coherencias

    # LATERALIZACIÓN
    resultados['lateralizacion']={

        'mu_C3_C4':

        resultados['C3']['mu']
        -
        resultados['C4']['mu'],

        'beta_C3_C4':

        resultados['C3']['beta']
        -
        resultados['C4']['beta']
    }

    return resultados

In [18]:
def calcular_CSP(
        X,
        y,
        n_components=2
    ):

    """
    X = [epochs, canales, muestras]

    y = etiquetas
    """

    csp=CSP(

        n_components=n_components,

        log=True,

        norm_trace=False
    )

    features=csp.fit_transform(
        X,
        y
    )

    return features,csp

In [13]:
archivo='Sujetos/S001/S001R04.edf'

raw = mne.io.read_raw_edf(

        archivo,

        preload=True,

        verbose=False
)

mne.datasets.eegbci.standardize(raw)

raw.notch_filter(60)

raw.filter(8,30)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1057 samples (6.606 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge

<RawEDF | S001R04.edf, 64 x 20000 (125.0 s), ~9.8 MB, data loaded>

In [22]:
metricas = obtener_metricas_eeg(raw)
metricas['C3']['spectral_entropy']

metricas['C3']['hjorth_activity']

metricas['C3']['hjorth_mobility']

metricas['C3']['hjorth_complexity']

np.float64(1.3096563468573348)

In [23]:
metricas['C3']['mu']

metricas['C3']['ERD_mu']

metricas['coherencia']['C3-C4']['coh_mu']

np.float64(0.5411316617615861)

In [26]:
fila={

'sujeto':'sub001',

'tarea':1,

# PROYECTO 1
'mu_C3':metricas['C3']['mu'],
'mu_Cz':metricas['Cz']['mu'],
'mu_C4':metricas['C4']['mu'],

'beta_C3':metricas['C3']['beta'],
'beta_Cz':metricas['Cz']['beta'],
'beta_C4':metricas['C4']['beta'],

'ERD_mu_C3':metricas['C3']['ERD_mu'],

'coh_mu_C3_C4':

metricas['coherencia']['C3-C4']['coh_mu'],

# PROYECTO 2
'entropy_C3':
metricas['C3']['spectral_entropy'],

'entropy_Cz':
metricas['Cz']['spectral_entropy'],

'entropy_C4':
metricas['C4']['spectral_entropy'],

'hjorth_activity_C3':
metricas['C3']['hjorth_activity'],

'hjorth_mobility_C3':
metricas['C3']['hjorth_mobility'],

'hjorth_complexity_C3':
metricas['C3']['hjorth_complexity']

}